# সহজ ভাষায় Notebook Guide

এই notebook **সম্পূর্ণ self-contained** — project folder লাগে না, internet-ও লাগে না (শুধু Section 8-এর FinBERT cell প্রথমবার model download করে)। সব ticker আর company **কাল্পনিক**; price একটা seeded random process থেকে বানানো, তাই যতবার চালান, হুবহু একই সংখ্যা আসবে। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। দরকার শুধু `numpy` আর `pandas`; `sqlite3`, `statistics`, `hashlib` Python-এর stdlib-এ আছে।
2. Section 8 (FinBERT) optional: `pip install transformers torch` লাগে, প্রথম run-এ ~440 MB download। এই cell এখানে চালানো হয়নি, তার output illustrative (ADR 0008)। বাকি notebook Section 9-এর pure-Python fallback দিয়ে চলে।
3. প্রতিটা metric-এর আগে অনুমান লিখুন (কোন ticker সবচেয়ে risky?), তারপর output-এর সাথে মেলান।

| Section | কী শিখব |
|---|---|
| 1 | Offline Snapshot: seeded synthetic OHLCV price |
| 2 | Simple বনাম log return |
| 3 | Volatility আর √252 |
| 4 | Maximum drawdown |
| 5 | Historical VaR, Expected Shortfall, fat tail |
| 6 | দুটো ফাঁদ: unadjusted split আর look-ahead |
| 7 | SQLite local store: schema, idempotent upsert, as-of query |
| 8 | FinBERT headline sentiment (optional, illustrative) |
| 9 | Pure-Python fallback scorer + sentiment cache |
| 10 | সময়ের সাথে sentiment aggregate: recency weight আর as-of filter |
| 11 | Risk input table — Class 32-এর কাঁচামাল |

In [1]:
import hashlib, math, re, sqlite3, tempfile
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
print("numpy", np.__version__, "| pandas", pd.__version__, "| SQLite", sqlite3.sqlite_version)

numpy 2.5.4 | pandas 3.1.0 | SQLite 3.50.4


## 1. Offline Snapshot — Seeded Synthetic Price

Live market data প্রতি মিনিটে বদলায়, network চায়, আর তার license-এর শর্ত থাকে। Agent আর test-এর জন্য দরকার এমন data যেটা **আজ আর ছয় মাস পরে একই** থাকে। তাই project একটা bundled snapshot পড়ে; এখানে সেটা আমরা নিজেরাই বানাই।

Model: geometric Brownian motion — প্রতিদিনের log return = drift + σ·√dt·z। তিনটা কাল্পনিক ticker তিন রকম চরিত্রের:

* **NOVX** — shock আসে Student-t (df=3) থেকে: বেশিরভাগ দিন শান্ত, মাঝে মাঝে বড় লাফ (**fat tail**)।
* **KTLX** — normal shock, কম σ: "শান্ত" stock।
* **ORBQ** — বেশি σ, আর একটা খারাপ খবরের দিনে −22% crash।

`pd.bdate_range` শুধু সোম-শুক্র দেয়; আসল exchange ছুটির দিনও বাদ দেয়, তাই বছরে ~252 trading day।

In [2]:
TICKERS = {   # all fictional
    "NOVX": dict(name="Nova Grid Energy",    start=48.0,  mu=0.08, sigma=0.32, shocks="student_t"),
    "KTLX": dict(name="Kite Logistics",      start=112.0, mu=0.05, sigma=0.18, shocks="normal"),
    "ORBQ": dict(name="Orbiq Semiconductor", start=27.0,  mu=0.12, sigma=0.45, shocks="normal",
                 crash_day=240, crash=-0.22),
}
DAYS = pd.bdate_range(end="2026-10-14", periods=260)   # Mon-Fri only
SEED = 10

def make_prices(spec, seed):
    rng = np.random.default_rng(seed)
    n, dt = len(DAYS), 1 / 252
    if spec["shocks"] == "student_t":
        z = rng.standard_t(df=3, size=n) / math.sqrt(3)            # t(3) has variance 3 -> rescale to 1
    else:
        z = rng.standard_normal(n)
    log_r = (spec["mu"] - 0.5 * spec["sigma"] ** 2) * dt + spec["sigma"] * math.sqrt(dt) * z
    if "crash_day" in spec:
        log_r[spec["crash_day"]] += math.log(1 + spec["crash"])     # one -22% day
    log_r[0] = 0.0
    close = spec["start"] * np.exp(np.cumsum(log_r))
    open_ = np.r_[spec["start"], close[:-1]] * np.exp(rng.normal(0, 0.003, n))   # overnight gap
    high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.006, n)))
    low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.006, n)))
    return pd.DataFrame({"date": DAYS.strftime("%Y-%m-%d"), "open": open_, "high": high, "low": low,
                         "close": close, "volume": rng.lognormal(13, 0.4, n).astype("int64")})

prices = pd.concat([make_prices(spec, SEED + i).assign(ticker=t) for i, (t, spec) in enumerate(TICKERS.items())],
                   ignore_index=True).round(4)
ok = ((prices.high >= prices[["open", "close"]].max(axis=1)) & (prices.low <= prices[["open", "close"]].min(axis=1))).all()
print(len(prices), "rows |", prices.date.min(), "->", prices.date.max(), "| OHLC consistent:", ok,
      "| ORBQ crash day:", DAYS[240].date())
prices.head(3)

780 rows | 2025-10-16 -> 2026-10-14 | OHLC consistent: True | ORBQ crash day: 2026-09-17


,date,open,high,low,close,volume,ticker
0,2025-10-16,48.1070,48.2125,47.6954,48.0000,334883,NOVX
1,2025-10-17,48.3809,48.4228,47.6831,48.1965,245774,NOVX
2,2025-10-20,48.3774,48.6859,47.9226,48.5805,322956,NOVX


## 2. Simple বনাম Log Return

Price নিজে তুলনাযোগ্য না (48 টাকার stock আর 112 টাকার stock), return তুলনাযোগ্য। দুই রকম:

* **Simple return** R = Pₜ/Pₜ₋₁ − 1 — "আজ কত % লাভ/ক্ষতি"। Portfolio-তে asset জুড়ে যোগ করা যায়, তাই VaR-এর টাকার ক্ষতি এতেই মাপি।
* **Log return** r = ln(Pₜ/Pₜ₋₁) — **সময় জুড়ে যোগ করা যায়**, তাই volatility আর statistics-এ সুবিধা।

ছোট উদাহরণ: 100 → 110 → 99। Simple return +10%, −10% — যোগফল 0%, অথচ আসলে 1% ক্ষতি।

In [3]:
p = pd.Series([100.0, 110.0, 99.0])
simple, logr = p.pct_change().dropna(), np.log(p).diff().dropna()
print("simple:", simple.round(4).tolist(), "sum =", round(simple.sum(), 4), "| real total =", round(p.iloc[-1] / p.iloc[0] - 1, 4))
print("log:   ", logr.round(4).tolist(), "sum =", round(logr.sum(), 4), "| ln(99/100) =", round(math.log(0.99), 4))

simple: [0.1, -0.1] sum = 0.0 | real total = -0.01
log:    [0.0953, -0.1054] sum = -0.0101 | ln(99/100) = -0.0101


In [4]:
px = prices.pivot(index="date", columns="ticker", values="close")
log_ret = np.log(px).diff().dropna()      # for volatility
simple_ret = px.pct_change().dropna()     # for VaR / ES: money lost is a simple return
print("largest daily |simple - log| gap:", (simple_ret - log_ret).abs().max().round(4).to_dict())
log_ret.tail(3).round(5)

largest daily |simple - log| gap: {'KTLX': 0.0004, 'NOVX': 0.0028, 'ORBQ': 0.0317}


ticker,KTLX,NOVX,ORBQ
date,,,
2026-10-12,-0.00690,0.02697,0.01982
2026-10-13,0.00702,0.00087,-0.02431
2026-10-14,0.01445,0.02418,0.00600


সাধারণ দিনে দুটো প্রায় সমান; পার্থক্য বড় হয় বড় move-এ — ORBQ-এর crash দিনে simple প্রায় −23%, log প্রায় −0.26: ফাঁক ~৩ percentage point। Rule: **একটা pipeline-এ একটা convention, আর memo-তে লিখে দিন কোনটা।**

## 3. Volatility আর √252

Volatility = daily log return-এর sample standard deviation, বছরে রূপান্তর করতে × √252:

σ_daily = √( Σ (rₜ − r̄)² / (n − 1) ),  σ_annual = σ_daily × √252

√252 কেন? Return যদি দিন-থেকে-দিন independent হয়, variance সময়ের সাথে যোগ হয়: ২৫২ দিনের variance = 252·σ², তাই std = √252·σ। এটা একটা **assumption** — volatility clustering থাকলে খাটে না (context.md §4.1)।

হাতে-কলমে ৫টা return: +1%, −2%, +0.5%, +1.5%, −1%। Mean 0; বর্গের যোগ 8.5 (%²); ÷4 = 2.125 → σ = 1.458%; × 15.87 = **23.1%**। সাবধান: `pandas.Series.std()` default `ddof=1`, `numpy.std()` default `ddof=0` — একই data, দুই উত্তর।

In [5]:
r = np.array([0.010, -0.020, 0.005, 0.015, -0.010])
print(f"pandas .std() (ddof=1): {pd.Series(r).std():.5f} | numpy .std() (ddof=0): {r.std():.5f}")
print(f"annualized: {pd.Series(r).std():.5f} x sqrt(252) = {pd.Series(r).std() * math.sqrt(252):.4f}")

pandas .std() (ddof=1): 0.01458 | numpy .std() (ddof=0): 0.01304
annualized: 0.01458 x sqrt(252) = 0.2314


In [6]:
vol = log_ret.std() * math.sqrt(252)
pd.DataFrame({"daily_std": log_ret.std(), "annual_vol": vol,
              "sigma_used_to_generate": pd.Series({t: s["sigma"] for t, s in TICKERS.items()})}).round(4)

,daily_std,annual_vol,sigma_used_to_generate
KTLX,0.0105,0.1672,0.18
NOVX,0.0172,0.2725,0.32
ORBQ,0.0316,0.5019,0.45


**পড়ার নিয়ম:** আমরা নিজেরা σ বসিয়েছি, তবু estimate মেলে না — এক বছরের data-য় volatility estimate-এর নিজস্ব error প্রায় σ/√(2n) ≈ ৪-৫%, আর NOVX-এর fat-tail shock-এ আরও বেশি। ORBQ-এর estimate বেশি এসেছে একটা crash দিনের জন্য: একটা দিন পুরো বছরের σ টেনে তোলে। মানে **risk metric নিজেই একটা অনিশ্চিত সংখ্যা** — memo-তে দুই দশমিকের বেশি precision দেখানো মিথ্যা নিশ্চয়তা।

## 4. Maximum Drawdown

Drawdown = আগের সর্বোচ্চ দাম থেকে এখন কত নিচে: DDₜ = Pₜ / max(P₀…Pₜ) − 1। **Maximum drawdown** = সবচেয়ে খারাপ DD। Volatility বলে "কত কাঁপে", drawdown বলে "সবচেয়ে খারাপ সময়ে peak থেকে কত হারিয়েছিলাম" — investor যেটা আসলে অনুভব করে।

In [7]:
p = pd.Series([100, 120, 90, 110, 130, 104.0])
pd.DataFrame({"price": p, "running_max": p.cummax(), "drawdown": (p / p.cummax() - 1).round(4)}).T

,0,1,2,3,4,5
price,100.0,120.0,90.00,110.0000,130.0,104.0
running_max,100.0,120.0,120.00,120.0000,130.0,130.0
drawdown,0.0,0.0,-0.25,-0.0833,0.0,-0.2


In [8]:
dd = px / px.cummax() - 1
mdd = dd.min()
trough = dd.idxmin()
pd.DataFrame({"max_drawdown": mdd.round(4), "peak_date": {t: px[t].loc[:trough[t]].idxmax() for t in px},
              "trough_date": trough})

,max_drawdown,peak_date,trough_date
KTLX,-0.1049,2025-11-12,2025-12-24
NOVX,-0.5078,2025-11-11,2026-10-09
ORBQ,-0.4154,2026-01-12,2026-10-02


120 → 90 = −25%, পরে 130-এ নতুন peak আর 104-এ −20%; সর্বোচ্চ −25%। Drawdown path-dependent: একই return গুলো অন্য ক্রমে এলে মান বদলায়, volatility বদলায় না।

## 5. Value at Risk, Expected Shortfall আর Fat Tail

* **Historical VaR (95%, 1-day)** = daily simple return-এর 5th percentile, ধনাত্মক করে: "১০০ দিনের ৯৫ দিনে ক্ষতি এর বেশি হয়নি"। বাকি ৫ দিনে **কত বেশি** — VaR বলে না।
* **Expected Shortfall (ES95)** = ওই সবচেয়ে খারাপ ৫% দিনের গড় ক্ষতি — tail-এর ভেতরের ছবি।
* **Parametric (normal) VaR** = −(μ − z·σ), z₀.₉₅ = 1.645, z₀.₉₉ = 2.326 — ধরে নেয় return normal।

২৫৯টা return-এর 5% মানে মাত্র ~১৩টা tail দিন; 99%-এ ~৩টা। Tail metric তাই খুব noisy।

In [9]:
Z = NormalDist().inv_cdf

def var_hist(r, level=0.95):
    return -np.quantile(r, 1 - level)

def es_hist(r, level=0.95):
    q = np.quantile(r, 1 - level)
    return -r[r <= q].mean()

def var_normal(r, level=0.95):
    return -(r.mean() - Z(level) * r.std(ddof=1))

rows = {}
for t in px:
    r = simple_ret[t].to_numpy()
    rows[t] = {"VaR95_hist": var_hist(r), "VaR95_normal": var_normal(r), "ES95_hist": es_hist(r),
               "VaR99_hist": var_hist(r, 0.99), "VaR99_normal": var_normal(r, 0.99),
               "excess_kurtosis": pd.Series(r).kurt(), "tail_days_95": int((r <= np.quantile(r, 0.05)).sum())}
risk = pd.DataFrame(rows).T.round(4)
risk

,VaR95_hist,VaR95_normal,ES95_hist,VaR99_hist,VaR99_normal,excess_kurtosis,tail_days_95
KTLX,0.0169,0.0167,0.0202,0.0214,0.0239,-0.2980,13.0
NOVX,0.0303,0.0303,0.0444,0.0516,0.0420,2.3823,13.0
ORBQ,0.0463,0.0502,0.0689,0.0625,0.0711,11.5864,13.0


In [10]:
position = 100_000
for t in px:
    print(f"{t}: on a {position:,} BDT position, 1-day 95% VaR = {position * risk.loc[t, 'VaR95_hist']:,.0f} BDT, "
          f"ES95 = {position * risk.loc[t, 'ES95_hist']:,.0f} BDT")

KTLX: on a 100,000 BDT position, 1-day 95% VaR = 1,690 BDT, ES95 = 2,020 BDT


NOVX: on a 100,000 BDT position, 1-day 95% VaR = 3,030 BDT, ES95 = 4,440 BDT
ORBQ: on a 100,000 BDT position, 1-day 95% VaR = 4,630 BDT, ES95 = 6,890 BDT


**পড়ার নিয়ম:**

* **NOVX** — 99%-এ historical VaR normal VaR-এর চেয়ে বড়: normal distribution fat tail-কে ছোট করে দেখায়। Excess kurtosis > 0 সেই চিহ্ন (normal-এ 0)।
* **ORBQ** — উল্টো ঘটনা: একটা −22% দিন σ-কে এত বাড়ায় যে normal VaR historical-এর চেয়েও বড় হয়, অথচ সেই crash-এর আসল আকার দেখা যায় শুধু ES আর kurtosis-এ। দুই রকম ভুল, একই কারণ: **একটা সংখ্যা tail-কে ধরতে পারে না**।
* VaR আর ES দুটোই memo-তে থাকলে পাঠক বোঝে "সাধারণ খারাপ দিন" আর "সত্যিকারের খারাপ দিন"-এর পার্থক্য।

## 6. দুটো ফাঁদ — Unadjusted Split আর Look-ahead

**(ক) Split:** 2-for-1 split-এ share সংখ্যা দ্বিগুণ, দাম অর্ধেক — investor-এর কোনো ক্ষতি নেই। কিন্তু unadjusted close-এ সেটা −50%-এর "crash" দেখায়। Live refresh-এ adjusted price নিন (yfinance-এ `auto_adjust` স্পষ্ট দিন) বা split factor দিয়ে নিজে adjust করুন।

In [11]:
raw = px["KTLX"].copy()
raw.iloc[200:] = raw.iloc[200:] / 2      # a 2-for-1 split that nobody adjusted
for name, s in (("adjusted", px["KTLX"]), ("unadjusted", raw)):
    print(f"{name:<11} max drawdown {(s / s.cummax() - 1).min():8.2%} | worst day {s.pct_change().min():8.2%} "
          f"| VaR95 {var_hist(s.pct_change().dropna().to_numpy()):.4f}")

adjusted    max drawdown  -10.49% | worst day   -2.58% | VaR95 0.0169
unadjusted  max drawdown  -55.23% | worst day  -49.68% | VaR95 0.0170


VaR প্রায় নড়েনি (একটা দিন মাত্র), কিন্তু drawdown আর worst day পুরো ভুল — তাই প্রতিটা metric আলাদা করে sanity-check দরকার।

**(খ) Look-ahead:** "জুলাইয়ের শেষে ORBQ কতটা risky ছিল?" — এর উত্তরে শুধু জুলাই পর্যন্ত data চলবে। পুরো file থেকে হিসাব করলে ভবিষ্যতের crash অতীতের risk-এ ঢুকে যায়। Backtest-এ এটাই সবচেয়ে সাধারণ, সবচেয়ে চুপচাপ bug।

In [12]:
as_of_early = DAYS[230].strftime("%Y-%m-%d")      # before ORBQ's crash day
honest = px["ORBQ"].loc[:as_of_early]
print(f"ORBQ max drawdown as of {as_of_early}, data up to that date only: {(honest / honest.cummax() - 1).min():.2%}")
print(f"same question answered from the full file (look-ahead):          {mdd['ORBQ']:.2%}")

ORBQ max drawdown as of 2026-09-03, data up to that date only: -31.52%
same question answered from the full file (look-ahead):          -41.54%


## 7. SQLite Local Store

Snapshot-কে একটা single-file database-এ রাখি: Python stdlib-এই আছে, server লাগে না, copy করলেই backup। চারটা table:

* `prices` — `PRIMARY KEY (ticker, date)`: একই দিন দুবার ঢুকতে পারে না — duplicate আটকানোর কাজ app code না, **DB constraint** করে।
* `news` — headline-এর নিজস্ব stable id (`NOVX-01`); agent পরে এই id cite করবে।
* `sentiment_cache` — `(text_hash, model)` key: একই headline একই model-এ দুবার score হয় না।
* `snapshot_meta` — version, as-of, source: memo-তে লেখা হবে "কোন data থেকে"।

Write হয় **upsert** দিয়ে (`INSERT ... ON CONFLICT DO UPDATE`): refresh দুবার চালালে row দ্বিগুণ হয় না — idempotent। `PRAGMA journal_mode=WAL` একজন writer চলাকালীন অন্যদের (API, UI) পড়তে দেয়। Demo DB একটা temp folder-এ, যাতে repo-তে file না জমে।

In [13]:
AS_OF = "2026-10-14T16:00"           # market close of the last snapshot day
DB = Path(tempfile.mkdtemp()) / "market_snapshot.db"
SCHEMA = '''
CREATE TABLE IF NOT EXISTS prices (ticker TEXT NOT NULL, date TEXT NOT NULL, open REAL, high REAL, low REAL,
                                   close REAL NOT NULL, volume INTEGER, PRIMARY KEY (ticker, date));
CREATE TABLE IF NOT EXISTS news (id TEXT PRIMARY KEY, ticker TEXT NOT NULL, published_at TEXT NOT NULL,
                                 source TEXT, headline TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS sentiment_cache (text_hash TEXT NOT NULL, model TEXT NOT NULL, label TEXT,
                                            p_pos REAL, p_neg REAL, p_neu REAL, PRIMARY KEY (text_hash, model));
CREATE TABLE IF NOT EXISTS snapshot_meta (key TEXT PRIMARY KEY, value TEXT);
'''
COLS = ["ticker", "date", "open", "high", "low", "close", "volume"]
UPSERT = ("INSERT INTO prices VALUES (?,?,?,?,?,?,?) ON CONFLICT(ticker, date) DO UPDATE SET "
          "open=excluded.open, high=excluded.high, low=excluded.low, close=excluded.close, volume=excluded.volume")

def connect(path):
    con = sqlite3.connect(path)
    con.execute("PRAGMA journal_mode=WAL")
    con.executescript(SCHEMA)
    return con

def upsert_prices(con, df):
    rows = df[COLS].itertuples(index=False, name=None)
    with con:                                                          # one transaction
        con.executemany(UPSERT, rows)

con = connect(DB)
for run in (1, 2):                    # a second refresh must not duplicate rows
    upsert_prices(con, prices)
    print(f"after refresh {run}: {con.execute('SELECT COUNT(*) FROM prices').fetchone()[0]} rows")

after refresh 1: 780 rows
after refresh 2: 780 rows


In [14]:
# The common shortcut: pandas append does not know "this day is already stored". The primary key catches it.
try:
    prices.head(2)[COLS].to_sql("prices", con, if_exists="append", index=False)
except pd.errors.DatabaseError as e:   # pandas 3 wraps the sqlite3 error
    print("to_sql append ->", type(e.__cause__).__name__, ":", e.__cause__)
print("rows still:", con.execute("SELECT COUNT(*) FROM prices").fetchone()[0])

to_sql append -> IntegrityError : UNIQUE constraint failed: prices.ticker, prices.date
rows still: 780


In [15]:
version = hashlib.sha256(prices.to_csv(index=False).encode()).hexdigest()[:12]   # content hash = snapshot version
with con:
    con.executemany("INSERT OR REPLACE INTO snapshot_meta VALUES (?, ?)",
                    [("version", version), ("as_of", AS_OF), ("source", f"synthetic GBM, seed={SEED}")])
con.close()

con = connect(DB)                      # a fresh connection: the data survived on disk
print(dict(con.execute("SELECT key, value FROM snapshot_meta").fetchall()))
q = "SELECT date, close FROM prices WHERE ticker = ? AND date <= ? ORDER BY date"
ktlx_h1 = pd.read_sql(q, con, params=("KTLX", "2026-06-30"))     # an as-of query: nothing after June
print(len(ktlx_h1), "KTLX rows up to 2026-06-30, last:", ktlx_h1.iloc[-1].to_dict())

{'version': 'cf0a46a2a5c0', 'as_of': '2026-10-14T16:00', 'source': 'synthetic GBM, seed=10'}
184 KTLX rows up to 2026-06-30, last: {'date': '2026-06-30', 'close': 126.1232}


Version হলো data-র content hash — একই data = একই version, একটা সংখ্যা বদলালেই নতুন version। Memo-তে এই version লেখা থাকলে পরে যে কেউ ঠিক সেই data দিয়ে memo-টা আবার বানাতে পারে।

## 8. Financial News + FinBERT Sentiment

প্রতিটা headline কাল্পনিক, নিজস্ব id আর timestamp সহ। লক্ষ্য করুন `ORBQ-03`: last snapshot দিনের market close-এর **পরে** প্রকাশিত — as-of-close score-এ এটা ঢুকবে না (Section 10)।

In [16]:
NEWS = [
    ("NOVX-01", "NOVX", "2026-09-28T13:10", "Wire A", "Nova Grid Energy beats quarterly revenue estimates as storage orders double"),
    ("NOVX-02", "NOVX", "2026-10-08T09:30", "Wire B", "Nova Grid Energy cuts full-year guidance on delayed grid contracts"),
    ("NOVX-03", "NOVX", "2026-10-12T11:00", "Wire A", "Nova Grid Energy shares flat ahead of utility regulator hearing"),
    ("KTLX-01", "KTLX", "2026-09-15T08:45", "Wire C", "Kite Logistics announces dividend increase of 5 percent"),
    ("KTLX-02", "KTLX", "2026-10-09T15:20", "Wire B", "Kite Logistics fuel costs rise but margins hold steady"),
    ("KTLX-03", "KTLX", "2026-10-13T10:05", "Wire A", "Kite Logistics not expected to miss delivery targets, CFO says"),
    ("ORBQ-01", "ORBQ", "2026-09-17T14:00", "Wire C", "Orbiq Semiconductor faces export probe; shares slide 22 percent"),
    ("ORBQ-02", "ORBQ", "2026-10-02T12:30", "Wire A", "Orbiq Semiconductor wins supply deal with major handset maker"),
    ("ORBQ-03", "ORBQ", "2026-10-14T18:05", "Wire B", "Orbiq Semiconductor recalls defective chip batch"),
]
with con:
    con.executemany("INSERT OR REPLACE INTO news VALUES (?,?,?,?,?)", NEWS)
news = pd.read_sql("SELECT * FROM news ORDER BY ticker, published_at", con)
news[["id", "published_at", "headline"]]

,id,published_at,headline
0,KTLX-01,2026-09-15T08:45,Kite Logistics announces dividend increase of ...
1,KTLX-02,2026-10-09T15:20,Kite Logistics fuel costs rise but margins hol...
2,KTLX-03,2026-10-13T10:05,Kite Logistics not expected to miss delivery t...
3,NOVX-01,2026-09-28T13:10,Nova Grid Energy beats quarterly revenue estim...
4,NOVX-02,2026-10-08T09:30,Nova Grid Energy cuts full-year guidance on de...
5,NOVX-03,2026-10-12T11:00,Nova Grid Energy shares flat ahead of utility ...
6,ORBQ-01,2026-09-17T14:00,Orbiq Semiconductor faces export probe; shares...
7,ORBQ-02,2026-10-02T12:30,Orbiq Semiconductor wins supply deal with majo...
8,ORBQ-03,2026-10-14T18:05,Orbiq Semiconductor recalls defective chip batch


### কেন FinBERT

FinBERT একটা **classifier**, LLM না: headline ঢোকে → BERT encoder → `[CLS]` vector → একটা linear layer → তিনটা logit → softmax → `positive / negative / neutral`-এর probability, যোগফল 1। কোনো sampling নেই, তাই একই headline-এ প্রতিবার একই উত্তর; আর probability সরাসরি একটা সংখ্যায় রূপ নেয় (`p_pos − p_neg`), যেটা পরে Risk Score-এ যাবে। LLM-কে "এটা positive না negative?" জিজ্ঞেস করলে পাওয়া যায় একটা শব্দ, ধীরে, আর প্রতিবার একই হবে তার নিশ্চয়তা নেই।

| Model | Use Case | Advantage | Limitation |
|---|---|---|---|
| **`ProsusAI/finbert`** (chosen, ADR 0009) | Financial headline-এর 3-class sentiment | Financial text-এ further pre-train + Financial PhraseBank-এ fine-tune; native positive/negative/neutral; CPU-তেও চলে | ~440 MB; 2019-এর BERT-base, 512 token সীমা; ছোট বাক্যে train, লম্বা article-এ না; শুধু English |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` (Week 10) | Tweet / social media sentiment | Native 3-class, students আগে থেকে চেনে | Tweet domain — "guidance cut", "beats estimates" জাতীয় finance ভাষায় domain shift |
| `mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis` | হালকা finance sentiment | DistilRoBERTa — ছোট, দ্রুত; Financial PhraseBank-এ fine-tune | Card-এর ~0.98 accuracy একই dataset-এর split-এ মাপা — আশাবাদী; community model, কম যাচাই করা |
| `llama3.1:8b` zero-shot (Ollama) | আলাদা model ছাড়া label | নতুন download নেই; কারণ ব্যাখ্যা করতে পারে | ধীর, non-deterministic, calibrated probability নেই, unit-test কঠিন |

Model একবার download হলে `HF_HUB_OFFLINE=1` দিয়ে নিশ্চিত করুন আর কখনো network-এ যাবে না।

⚠️ **নিচের cell এখানে চালানো হয়নি** (ADR 0008: ~440 MB download, torch লাগে)। নিজের machine-এ চালান।

In [ ]:
# HIGHLIGHTS: FinBERT একটা deterministic classifier — একই headline-এ একই probability, তাই Risk Score
# reproducible থাকে। Model প্রথম call-এ download হয় (~440 MB), তারপর local cache থেকে। top_k=None দিলে
# তিনটা label-এরই probability আসে (শুধু সেরাটা না) — net sentiment = p_pos - p_neg বানাতে দুটোই লাগে।
# batch_size headline-গুলো একসাথে চালায়; truncation=True 512-token-এর বেশি text কেটে দেয় (না দিলে error)।
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)

def finbert_scorer(texts):
    out = finbert(list(texts), batch_size=8, truncation=True)
    return [{d["label"]: d["score"] for d in scores} for scores in out]

for (hid, *_, headline), p in zip(NEWS, finbert_scorer([n[-1] for n in NEWS])):
    print(f"{hid}  {max(p, key=p.get):<8}  " + "  ".join(f"{k}={v:.3f}" for k, v in sorted(p.items())), "|", headline[:50])

**Illustrative output — measured না।** নিজের machine-এ চালালে প্রতিটা headline-এর জন্য তিনটা probability দেখবেন (যোগফল 1)। যা খেয়াল করবেন:

* "beats estimates", "dividend increase", "wins supply deal" — positive সবচেয়ে বেশি হওয়ার কথা; "cuts guidance", "export probe", "recalls defective" — negative।
* "shares flat ahead of hearing" — neutral; finance headline-এর বড় অংশই আসলে neutral।
* **`KTLX-03`** ("not expected to miss") আর **`KTLX-02`** ("costs rise but margins hold") — negation আর মিশ্র খবর; এখানে model কী বলে, সেটাই আসল পরীক্ষা।
* Probability-কে নিশ্চিত সত্য ভাববেন না: fine-tuned model প্রায়ই overconfident (0.95+ এমনকি ভুল উত্তরেও)।

## 9. Pure-Python Fallback Scorer + Sentiment Cache

FinBERT না থাকলেও notebook চলবে, তাই একটা ছোট word-list scorer — **এটা model না, শুধু teaching fallback।** Output-এর আকার FinBERT-এর মতো (তিনটা probability), তাই বাকি code বদলাতে হয় না: scorer বদলান, pipeline একই।

In [17]:
POS = {"beats", "beat", "wins", "win", "increase", "raises", "record", "growth", "upgrade", "double", "steady", "deal"}
NEG = {"cuts", "cut", "miss", "misses", "probe", "slide", "slides", "recall", "recalls", "defective", "delayed",
       "downgrade", "lawsuit", "falls"}

def lexicon_scorer(texts):
    out = []
    for text in texts:
        words = re.findall(r"[a-z]+", text.lower())
        raw = {"positive": sum(w in POS for w in words) + 0.25, "negative": sum(w in NEG for w in words) + 0.25,
               "neutral": 0.75}
        total = sum(raw.values())
        out.append({k: v / total for k, v in raw.items()})
    return out

SCORER, MODEL_NAME = lexicon_scorer, "lexicon-demo-v1"   # after running Section 8: finbert_scorer, "ProsusAI/finbert"

In [18]:
def score_cached(con, texts, scorer, model_name):
    # Score only headlines this model has not scored before; one batched call for the rest.
    keys = [hashlib.sha256(t.encode("utf-8")).hexdigest() for t in texts]
    marks = ",".join("?" * len(keys))
    known = {row[0]: row[1:] for row in con.execute(
        f"SELECT text_hash, p_pos, p_neg, p_neu FROM sentiment_cache WHERE model = ? AND text_hash IN ({marks})",
        [model_name, *keys])}
    todo = [(k, t) for k, t in zip(keys, texts) if k not in known]
    if todo:
        fresh = scorer([t for _, t in todo])
        rows = [(k, model_name, max(p, key=p.get), p["positive"], p["negative"], p["neutral"]) for (k, _), p in zip(todo, fresh)]
        with con:
            con.executemany("INSERT OR REPLACE INTO sentiment_cache VALUES (?,?,?,?,?,?)", rows)
        known.update({r[0]: r[3:] for r in rows})
    return [dict(zip(("p_pos", "p_neg", "p_neu"), known[k])) for k in keys], len(todo)

for attempt in (1, 2):
    scores, n_scored = score_cached(con, news.headline.tolist(), SCORER, MODEL_NAME)
    print(f"run {attempt}: model called for {n_scored} of {len(news)} headlines")

scored = pd.concat([news, pd.DataFrame(scores)], axis=1)
scored["net"] = scored.p_pos - scored.p_neg
scored[["id", "headline", "p_pos", "p_neg", "p_neu", "net"]].round(3)

run 1: model called for 9 of 9 headlines


run 2: model called for 0 of 9 headlines


,id,headline,p_pos,p_neg,p_neu,net
0,KTLX-01,Kite Logistics announces dividend increase of ...,0.556,0.111,0.333,0.444
1,KTLX-02,Kite Logistics fuel costs rise but margins hol...,0.556,0.111,0.333,0.444
2,KTLX-03,Kite Logistics not expected to miss delivery t...,0.111,0.556,0.333,-0.444
3,NOVX-01,Nova Grid Energy beats quarterly revenue estim...,0.692,0.077,0.231,0.615
4,NOVX-02,Nova Grid Energy cuts full-year guidance on de...,0.077,0.692,0.231,-0.615
5,NOVX-03,Nova Grid Energy shares flat ahead of utility ...,0.200,0.200,0.600,0.000
6,ORBQ-01,Orbiq Semiconductor faces export probe; shares...,0.077,0.692,0.231,-0.615
7,ORBQ-02,Orbiq Semiconductor wins supply deal with majo...,0.692,0.077,0.231,0.615
8,ORBQ-03,Orbiq Semiconductor recalls defective chip batch,0.077,0.692,0.231,-0.615


**পড়ার নিয়ম:**

* দ্বিতীয় run-এ model একবারও ডাকা হয়নি — cache key = (headline-এর hash, model name)। Model বা তার version বদলালে key বদলায়, পুরোনো score আর ব্যবহার হয় না।
* `KTLX-03` "not **expected to miss**" — word list "miss" দেখে negative বলেছে, অথচ খবরটা ভালো। `KTLX-02`-এ "costs rise" খারাপ, "hold steady" ভালো — word list শুধু "steady" দেখেছে। Context ছাড়া sentiment এভাবেই ভাঙে; Loughran-McDonald (2011) দেখিয়েছেন সাধারণ sentiment dictionary finance text-এ নিয়মিত ভুল করে।

## 10. সময়ের সাথে Sentiment Aggregate

একটা ticker-এর অনেক headline থেকে একটা সংখ্যা চাই:

* প্রতিটা headline-এর score sᵢ = p_pos − p_neg ∈ [−1, 1]।
* পুরোনো খবরের ওজন কম: wᵢ = 0.5^(ageᵢ / half-life) — half-life ৩০ দিন মানে এক মাসের পুরোনো খবরের ওজন অর্ধেক (project-এর default)। ছোট half-life নতুন খবরে দ্রুত সাড়া দেয়, কিন্তু একটা headline-এই দোলে।
* Net sentiment = Σ wᵢsᵢ / Σ wᵢ।
* **As-of filter:** published_at > as_of হলে বাদ — নইলে look-ahead।

হাতে-কলমে: ২ দিন আগের s = +0.80, ১০ দিন আগের s = −0.70, ১ দিন আগের s = −0.10 — দুই রকম half-life-এ।

In [19]:
HALF_LIFE_DAYS = 30.0

def net_sentiment(rows, as_of, half_life=HALF_LIFE_DAYS):
    t0 = pd.Timestamp(as_of)
    used = [r for r in rows if pd.Timestamp(r["published_at"]) <= t0]          # no look-ahead
    if not used:
        return 0.0, 0
    age = np.array([(t0 - pd.Timestamp(r["published_at"])).total_seconds() / 86400 for r in used])
    w = 0.5 ** (age / half_life)
    s = np.array([r["p_pos"] - r["p_neg"] for r in used])
    return float((w * s).sum() / w.sum()), len(used)

hand = [{"published_at": "2026-10-12T16:00", "p_pos": 0.85, "p_neg": 0.05},
        {"published_at": "2026-10-04T16:00", "p_pos": 0.10, "p_neg": 0.80},
        {"published_at": "2026-10-13T16:00", "p_pos": 0.20, "p_neg": 0.30}]
for h in (30.0, 7.0):
    print(f"half-life {h:>4} days: weights {[round(0.5 ** (a / h), 4) for a in (2, 10, 1)]} -> "
          f"net {net_sentiment(hand, AS_OF, h)[0]:+.4f}")
print("plain mean (no recency):", round(np.mean([0.80, -0.70, -0.10]), 4))

half-life 30.0 days: weights [0.9548, 0.7937, 0.9772] -> net +0.0406
half-life  7.0 days: weights [0.8203, 0.3715, 0.9057] -> net +0.1457
plain mean (no recency): 0.0


In [20]:
agg = {}
for t, g in scored.groupby("ticker"):
    net, n_used = net_sentiment(g.to_dict("records"), AS_OF)
    agg[t] = {"net_sentiment": round(net, 4), "headlines_used": n_used, "excluded_after_as_of": len(g) - n_used}
pd.DataFrame(agg).T

,net_sentiment,headlines_used,excluded_after_as_of
KTLX,0.0800,3.0,0.0
NOVX,-0.0433,3.0,0.0
ORBQ,0.1052,2.0,1.0


ORBQ-এর recall খবর (close-এর পরে) বাদ পড়েছে — কালকের memo-তে ঢুকবে, আজকেরটায় না। KTLX-এর তিনটা headline-ই আসলে ভালো খবর; `KTLX-03` ঠিক label পেলে net হতো +0.44, ভুল label-এ +0.08। সবচেয়ে নতুন headline-এর ওজন সবচেয়ে বেশি, তাই একটা ভুল নতুন label সবচেয়ে বেশি ক্ষতি করে। আর ৩টা headline-এর net sentiment একটা ভঙ্গুর সংখ্যা: খবর কম হলে 0-এর দিকে টানা (`net × n / (n + k)`) একটা সাধারণ উন্নতি — Brain Teaser 4।

## 11. Risk Input Table — Class 32-এর কাঁচামাল

প্রতিটা ticker-এর জন্য একটা row: চারটা সংখ্যা, সবই code-এ, সবই এই snapshot থেকে। পরের ক্লাসে এগুলো normalize আর weight করে **Risk Score** হবে, আর agent-রা শুধু সেটা ব্যাখ্যা করবে।

In [21]:
inputs = pd.DataFrame({
    "volatility": vol, "max_drawdown": -mdd, "var_95": risk["VaR95_hist"], "es_95": risk["ES95_hist"],
    "sentiment": pd.DataFrame(agg).T["net_sentiment"].astype(float),
}).round(4)
inputs.assign(snapshot=version, as_of=AS_OF).to_sql("risk_inputs", con, if_exists="replace", index_label="ticker")
con.close()
inputs

,volatility,max_drawdown,var_95,es_95,sentiment
KTLX,0.1672,0.1049,0.0169,0.0202,0.0800
NOVX,0.2725,0.5078,0.0303,0.0444,-0.0433
ORBQ,0.5019,0.4154,0.0463,0.0689,0.1052


## 📝 Key Takeaways

* **Offline Snapshot:** data একবার ingest, তারপর সব analysis local আর reproducible; "real-time" মানে ingestion-এর গতি, analysis-এর না।
* **Return:** টাকার ক্ষতিতে simple, সময় জুড়ে statistics-এ log; একটা convention, স্পষ্ট লেখা।
* **Volatility** = std(log return) × √252 — √252 i.i.d. assumption; `ddof` মিলিয়ে নিন।
* **Max drawdown** = peak থেকে সবচেয়ে খারাপ পতন; path-dependent।
* **VaR** বলে কত দিনে কতটা পর্যন্ত, **ES** বলে বাকি দিনে কত খারাপ; normal assumption fat tail লুকায়।
* **ফাঁদ:** unadjusted split, look-ahead, ছোট sample-এর tail metric।
* **SQLite:** primary key + upsert = idempotent refresh; snapshot version = content hash।
* **FinBERT:** deterministic 3-class classifier, probability → সংখ্যা; cache key-তে model name।
* **Aggregation:** recency weight + as-of filter; কম headline = কম ভরসা।

## 🧠 Brain Teasers

1. Section 1-এ `SEED` বদলে তিনবার চালান। NOVX-এর annual vol কতটা নড়ে? Memo-তে "27.25% volatility" লেখা কি সৎ?
2. Section 5-এ NOVX-এর `df=3`-কে `df=30` করুন (প্রায় normal)। VaR99 historical আর normal-এর ফাঁক কী হয়? Kurtosis?
3. Section 6-এর look-ahead cell-এ `as_of_early` crash-এর পরের দিন করুন। দুই সংখ্যা কি মিলে যায়? কেন?
4. Section 10-এ `net × n / (n + 3)` shrinkage যোগ করুন। কোন ticker-এর net সবচেয়ে বেশি বদলায়?
5. FinBERT চালাতে পারলে Section 8 আর 9-এর label মিলিয়ে দেখুন। কোন headline-এ দুটো আলাদা? কোনটা ঠিক?
6. Syndication-এ একই headline দুটো wire থেকে এলে net sentiment-এ কী হয়? কীভাবে dedupe করবেন?